# Speech-to-Text (STT) with Whisper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nontgcob/HKU-InnoWing-STT-TTS-Workshop/blob/9895b5c/src/stt.ipynb)

This notebook demonstrates Speech-to-Text (STT), also known as Automatic Speech Recognition (ASR), with the Whisper model from Hugging Face Transformers.

This version focuses on transcribing an audio file from the notebook runtime using a simple preprocessing and inference pipeline.


In [ ]:
%pip install -q "transformers>=5.0.0" "torch>=2.7.1" "librosa>=0.11.0" "soundfile>=0.13.1" "sentencepiece>=0.2.1"

## 1. Load the Whisper Model

We will use `openai/whisper-small`, which is a good balance between speed and transcription quality for a workshop notebook.


In [ ]:
from pathlib import Path

import librosa
import soundfile as sf
from IPython.display import Audio, display
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


def load_audio(path, target_sr=16000):
    audio, sample_rate = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sample_rate != target_sr:
        audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=target_sr)
    return audio


def transcribe_audio_file(path, sampling_rate=16000):
    audio = load_audio(path, target_sr=sampling_rate)
    input_features = processor.feature_extractor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt",
    ).input_features
    predicted_ids = model.generate(input_features)
    transcription = processor.tokenizer.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]
    return transcription


## 2. Transcribe an Audio File

Upload a WAV file to the notebook runtime or change the path below to point to your own audio file.


In [ ]:
audio_file_path = Path("recording.wav")

if audio_file_path.exists():
    file_transcription = transcribe_audio_file(audio_file_path)
    print("--- Transcription Result from Audio File ---")
    print(file_transcription)
    display(Audio(str(audio_file_path)))
else:
    print(
        "Upload a file named 'recording.wav' or change audio_file_path to "
        "your own WAV file before running this cell."
    )
